In [9]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
from scipy.stats import randint
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

| Column Name | Description | Type |
|-------------|-------------|------|
| **Time** | The seconds elapsed between this transaction and the first transaction in the dataset | Numerical (Integer) |
| **V1, V2, ..., V28** | 28 anonymized features resulting from Principal Component Analysis (PCA) transformation to protect sensitive cardholder information | Numerical (Integer/Float) |
| **Amount** | The transaction amount | Numerical (Integer/Float) |
| **Class** | **Target variable:** `1` = fraudulent transaction, `0` = legitimate transaction | Numerical (Binary: 0 or 1) |

In [6]:
# https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
data = pd.read_csv('../datasets/creditcard.csv')
data.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [4]:
data.shape

(284807, 31)

In [13]:
# data split
X = data.drop('Class', axis=1)
y = data['Class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# search space
param_dist = {
    'n_estimators': randint(50, 300),
    'max_depth': [5, 10, 15, None],
    'min_samples_split': randint(2, 15),
    'min_samples_leaf': randint(1, 10)
}

# pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(random_state=42, oob_score=True))
])

# hyperparameter tuning (search space)
param_dist = {
    'rf__n_estimators': [50, 100],
    'rf__max_depth': [5, 10, 20, None],
    'rf__min_samples_split': [2, 5, 10],
    'rf__min_samples_leaf': [1, 2, 4],
    # 'rf__max_features': ['sqrt', 'log2', None],
    # 'rf__class_weight': ['balanced', 'balanced_subsample', None]
}

# cv
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# model based on best estimators
random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_dist,
    n_iter=10, # to make the process faster reduced number of iterations
    cv=cv,
    scoring='accuracy',
    n_jobs=4, # not a big difference between -1 and 4 in timing so i changed it to 4 (the i/o might be bottleneck)
    random_state=42,
    verbose=1
)

# finding best params (hyperparameter tuning) and train final model/fit
random_search.fit(X_train, y_train)

# model (best model)
model = random_search.best_estimator_

# predict
y_pred = model.predict(X_test)

print(f"Best params: {random_search.best_estimator_}")
print(f"Best CV score (on training): {random_search.best_score_:.4f}")

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best params: Pipeline(steps=[('scaler', StandardScaler()),
                ('rf',
                 RandomForestClassifier(max_depth=10, n_estimators=50,
                                        oob_score=True, random_state=42))])
Best CV score (on training): 0.9995


In [15]:
y_pred_train = model.predict(X_train)

In [19]:
target_names = ['No Fraud', 'Fraud']

print("\n" + "="*60)
print("MODEL EVALUATION")
print("="*60)

# accuracy
test_accuracy = accuracy_score(y_test, y_pred)
train_accuracy = accuracy_score(y_train, y_pred_train)
gap = train_accuracy - test_accuracy
print(f"Training Accuracy: {train_accuracy:.2%}")
print(f"Test Accuracy: {test_accuracy:.2%}")
print(f"OOB Score (compare with test accuracy): {model.named_steps['rf'].oob_score_:.2%}")
print(f"Overfitting Gap: {gap:.4f}")


if gap < 0.01:
    print("✅ NO OVERFITTING (gap < 1%)")
    status = "Balanced"
elif gap < 0.02:
    print("✅ MINIMAL OVERFITTING (1-2%)")
    status = "Slight Overfitting"
elif gap < 0.03:
    print("⚠️  MILD OVERFITTING (2-3%)")
    status = "Mild Overfitting"
elif gap < 0.05:
    print("⚠️  MODERATE OVERFITTING (3-5%)")
    status = "Moderate Overfitting"
elif gap < 0.10:
    print("🔴 SIGNIFICANT OVERFITTING (5-10%)")
    status = "Significant Overfitting"
else:
    print("🔴 SEVERE OVERFITTING (>10%)")
    status = "Severe Overfitting"

if train_accuracy < 0.70 and test_accuracy < 0.70:
    print("📉 UNDERFITTING DETECTED: Model is too simple")
elif train_accuracy < 0.75:
    print("⚠️  Possible underfitting (training accuracy < 75%)")
else:
    print("✅ No underfitting detected")

# detailed classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=target_names))

# confusion matrix
cm = confusion_matrix(y_test, y_pred)
correct_predictions = np.trace(cm)
total_predictions = np.sum(cm)
accuracy = correct_predictions / total_predictions

print("\nConfusion Matrix:")
print(cm)


print(f"Correct predictions: {correct_predictions}")
print(f"Total predictions: {total_predictions}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Accuracy: {accuracy*100:.2f}%")

for i in range(cm.shape[0]):
    class_correct = cm[i, i]
    class_total = np.sum(cm[i, :])
    accuracy = class_correct / class_total
    print(f"{target_names[i]} Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")



MODEL EVALUATION
Training Accuracy: 99.97%
Test Accuracy: 99.95%
OOB Score (compare with test accuracy): 99.95%
Overfitting Gap: 0.0002
✅ NO OVERFITTING (gap < 1%)
✅ No underfitting detected

Classification Report:
              precision    recall  f1-score   support

    No Fraud       1.00      1.00      1.00     85295
       Fraud       0.97      0.76      0.85       148

    accuracy                           1.00     85443
   macro avg       0.98      0.88      0.93     85443
weighted avg       1.00      1.00      1.00     85443


Confusion Matrix:
[[85291     4]
 [   35   113]]
Correct predictions: 85404
Total predictions: 85443
Accuracy: 0.9995
Accuracy: 99.95%
No Fraud Accuracy: 1.0000 (100.00%)
Fraud Accuracy: 0.7635 (76.35%)


In [26]:
print("\n" + "="*60)
print("FEATURE IMPORTANCE")
print("="*60)

feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.named_steps['rf'].feature_importances_
}).sort_values('Importance', ascending=False)

feature_importance


FEATURE IMPORTANCE


,Feature,Importance
17,V17,0.192456
14,V14,0.181278
12,V12,0.121363
10,V10,0.085886
16,V16,0.073091
7,V7,0.035183
11,V11,0.034952
9,V9,0.034948
4,V4,0.033966
18,V18,0.021612
